# Paper Atypicality (z-scores) — Uzzi journal-pair atypicality on the Dimensions graph

Uzzi et al. (2013) atypicality for each paper, from the **journal pairs** of the papers it cites,
against a year-preserving rewiring null. The statistical model is the reference notebook's; the data
structures are not — the reference materialises Python dicts of edge lists over a decade of MAG, which
is not possible against a ~2.2e9-edge OpenAlex snapshot. Primary key: `paper_id`.

## Input caches — the files `oa.load_csr()` and `oa.build_journal()` actually read
```
Dimensions/cache/paper_csr.npz             # dim.CSR_NPZ    -> out_ptr, out_idx (references), in_ptr, in_idx, year, uni_mag
Dimensions/cache/paper_journal.parquet     # dim.JOURNAL_PQ -> work_id, source_id, journal, is_journal
Dimensions/cache/pub_year_source_map.npz   # dim.MAP_NPZ   -> what build_journal() reads
```
All three are produced by `notebook/references_w_year.ipynb` plus `dim.build_csr()`; build them
before running this notebook. The engine, its null model and its validation tests are the OpenAlex
notebook's, verbatim; only the data layer (`dim` for `oa`, `jour.` for `S`) differs.

## What the metric is

**Focal papers** are journal-type works published in `YEAR_RANGE`. The reference selects
`DocType == 'Journal'`; the Dimensions analogue is a publication whose `source_id` has
`source_titles.type == 'journal'`, which is `is_journal` in `dim.JOURNAL_PQ`.

**Observed counts.** For each focal paper, every unordered pair `i < j` of its references' journals is
counted **with multiplicity** into a per-focal-year counter: references in journals `[A, A, A, B, B]`
contribute `(A,A) = 3`, `(A,B) = 6`, `(B,B) = 1`. Self-pairs are real and are kept. This is what the
reference's `cal_J` does. A reference whose cited work has **no journal mapping is dropped before
pairing**, so it neither pairs with itself nor with a journal-bearing reference — the reference
requires *both* endpoints to be in its `MAGID -> JID` map.

**The null.** The donor pool for a focal year `cy` is the complete input-graph `(citing year cy, cited
year dy)` cell: every edge out of every paper published in `cy`, grouped by the cited paper's year —
not just the focal cohort's edges, and not just edges into journals. The reference permutes that cell's
cited column and reads the focal papers' slots back off it. A uniform permutation of the column is
exchangeable, so the focal slots are a simple random sample **without replacement** from the cell's
cited pool; drawing only those slots is exact, not an approximation, and is what makes the null
affordable here. `NUM_SHUFFLES` independent draws are made per focal year.

**Every eligible focal paper participates in the null** — every paper whose reference count is within
`[MIN_REFS, MAX_REFS]` — including a paper whose observed references carry no journal at all. Such a
paper contributes nothing to the observed counts, but rewiring can hand it journal-bearing references,
and the reference counts those. Restricting the null to papers that happen to have an observed journal
pair biases every null mean downward. Cited works with no journal mapping stay **in** the rewiring pool
as donors; they simply form no pair once assigned.

**The z-score.** Per journal pair per focal year, `z = (observed - null_mean) / null_std`, with
`null_std` the population standard deviation (`np.std`, `ddof=0`). Pairs whose `null_std == 0` are
**omitted**, as in the reference — no epsilon smoothing. Per paper, the median, 10th percentile and
minimum are taken over the z of the **distinct** pairs that paper touches (`Paper_pair` is a set in the
reference), which is why `n_pairs` is far smaller than a paper's contribution to the counters.

## Year range

| mode | range | how |
|---|---|---|
| `reference` (**default**) | 1990–2000 | MAG-compatible validation run |
| `full` | 1900–2020 | full production |
| explicit | anything | `NB_Z_YEARS=2000:2005` |

`NB_Z_YEARS` wins over `NB_Z_MODE`. Any range other than the full one **suffixes both outputs**
(`paper_z_score_1990_2000.parquet`), because `paper_validation` reads a bare `paper_z_score.parquet`
and would treat a decade of papers as the whole corpus without saying so. Promote a partial run
deliberately, by renaming it.

```
cd '/project/jevans/Dawoon/Science of Science/jobs/Dimensions'
sbatch --export=ALL,NB=paper_z_score,NB_Z_MODE=reference   -J dim_paper_z_score_ref  nb.sbatch
sbatch --export=ALL,NB=paper_z_score,NB_Z_MODE=full        -J dim_paper_z_score_full nb.sbatch
sbatch --export=ALL,NB=paper_z_score,NB_Z_YEARS=2000:2005  -J dim_paper_z_score_p    nb.sbatch
```

## Memory — the cohort is walked twice, on purpose

Pass one counts journal pairs; the null is drawn; pass two rebuilds **one paper's** distinct-pair
set at a time to aggregate the z it touches, then drops it.

Holding a set per paper until every pair's z was known is what made the 2010s unrunnable: a
paper with *m* journal-bearing references holds up to *m(m+1)/2* tuples, and at 3.7M focal
papers that is hundreds of GB. Measured — the 2019-2021, 2022-2023 and 2024-2025 chunks were
**OOM-killed at 250 GB**, and 2016-2018 sat at 243 GB after 13.5 h without finishing a single
year. On an isolated benchmark the rewrite cut peak memory **1.86 GB → 0.22 GB**.

It changes memory and nothing else, verified twice: bit-identical `work_df` and `pair_df` on a
synthetic graph, and on the real 2001-2006 range the rewritten run reproduced the earlier output
**byte for byte** — same SHA256, 5,555,275 papers and 41,793,371 pair-year rows, empty symmetric
difference both ways — at +1.6% runtime.

The remaining ceiling is `null_counts`: ten pair counters held at once, about ten times the size
of `obs`. At 2016-2018 pair volumes that is ~37 GB, comfortable inside 250 GB.

## Reproducibility

Randomness is seeded **per focal year** from `(SEED, focal_year)`, so a year's result depends on the
year, the input caches and the parameters — not on which other years were in the run or in what order
they were processed. Re-running 2000 alone reproduces 2000 from a full-range run. This is *not* bitwise
comparable with the MAG notebook, which calls `random.shuffle()` with no seed at all; the claim is
distributional equivalence of the null plus reproducibility of this implementation.

## Output (two files, `{SUFFIX}` empty only for the full range)
- `Dimensions/output/paper_z_score{SUFFIX}.parquet` — `paper_id, Z_median, Z_10pct, Z_min, n_pairs`
- `Dimensions/output/z_score_pair{SUFFIX}.parquet` — `code_1, code_2, year, Z_score`

`Z_min` is an intentional extension; the reference reports only the median and the 10th percentile.

In [1]:
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim
ROOT = dim.BASE; OUT = dim.OUT
print('dump:', dim.ROOT)

NUM_SHUFFLES = 10            # the reference builds 10 randomized networks
MIN_REFS, MAX_REFS = 2, 1000 # the reference skips >1000 refs outright; <2 cannot form a pair
SEED = 42                    # per-focal-year streams are derived from this, see _rng_for_year

# The reference computes 1990-2000. That is the default here too, so a plain run is the
# MAG-comparable one; the full corpus is opt-in because the shuffle null costs ~15 min/year.
REFERENCE_RANGE = (1990, 2000)
FULL_RANGE      = (1900, 2025)
# The range the project actually works in. 1900-1979 is a long tail of small cohorts that costs
# little but adds little; 1980 is where the corpus is dense enough for a cohort-level statistic.
EXTENDED_RANGE  = (1980, 2025)

# NB_Z_YEARS (explicit) beats NB_Z_MODE (named).
#   sbatch --export=ALL,NB=paper_z_score,NB_Z_MODE=reference  -J dim_paper_z_score_ref  nb.sbatch
#   sbatch --export=ALL,NB=paper_z_score,NB_Z_MODE=full       -J dim_paper_z_score_full nb.sbatch
#   sbatch --export=ALL,NB=paper_z_score,NB_Z_MODE=extended   -J dim_paper_z_score_ext  nb.sbatch
#   sbatch --export=ALL,NB=paper_z_score,NB_Z_YEARS=2000:2005 -J dim_paper_z_score_p    nb.sbatch
#
# A YEAR RANGE IS SAFE TO SPLIT. Randomness is seeded per focal year (_rng_for_year) and every
# z is computed inside the focal-year loop, so 1980:1989 and 1990:1999 run separately give
# exactly what 1980:1999 would -- verified on a synthetic graph, where one year run alone
# matched the same year inside a wider range for all 348 papers compared. Splitting is how the
# expensive later years get failure isolation.
_mode = os.environ.get('NB_Z_MODE', 'reference').strip().lower()
_yr   = os.environ.get('NB_Z_YEARS')
if _yr:
    YEAR_RANGE = tuple(int(x) for x in _yr.replace('-', ':').split(':'))
elif _mode == 'full':
    YEAR_RANGE = FULL_RANGE
elif _mode == 'reference':
    YEAR_RANGE = REFERENCE_RANGE
elif _mode == 'extended':
    YEAR_RANGE = EXTENDED_RANGE
else:
    raise ValueError(f"Unknown NB_Z_MODE={_mode!r}; expected 'reference', 'extended' or "
                     f"'full' (or set NB_Z_YEARS=<begin>:<end> for an explicit range)")
if len(YEAR_RANGE) != 2:
    raise ValueError(f'YEAR_RANGE must be (begin, end); got {YEAR_RANGE!r}')
if YEAR_RANGE[0] > YEAR_RANGE[1]:
    raise ValueError(f'YEAR_RANGE begin > end: {YEAR_RANGE!r}')

# The outputs of anything short of the full range are SUFFIXED. paper_validation reads a bare
# paper_z_score.parquet and would happily treat a decade of papers as the whole corpus, and
# nothing in its output would say otherwise -- so a partial file must not be able to take that
# name by accident. Promote a partial run deliberately, by renaming it.
SUFFIX = '' if tuple(YEAR_RANGE) == FULL_RANGE else f'_{YEAR_RANGE[0]}_{YEAR_RANGE[1]}'
print(f'mode {_mode!r}  YEAR_RANGE {YEAR_RANGE}'
      + (f'  -> outputs suffixed "{SUFFIX}"' if SUFFIX else '  (full range, unsuffixed)'))

# Everything below is fed by notebook/references_w_year.ipynb: it writes the per-publication
# map (year + source), the scalar and author parts, and the edge table with both years and both
# source ids. Build it once before running this notebook.
assert dim.have_consolidated(), (
    'run notebook/references_w_year.ipynb first -- it builds the map, the scalar parts and the edge table')
dim.summary()

dump: /project/jevans/dimensions/dimensions/dimensions_june_2025
mode 'reference'  YEAR_RANGE (1990, 2000)  -> outputs suffixed "_1990_2000"
dump     : /project/jevans/dimensions/dimensions/dimensions_june_2025
cache    : /project/jevans/Dawoon/Science of Science/Dimensions/cache
output   : /project/jevans/Dawoon/Science of Science/Dimensions/output
  consolidated edge table: present
  map            2.80 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_year_source_map.npz
  graph         19.00 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz
  csr           21.49 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
  journal        1.24 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_journal.parquet
  fos            0.77 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_fos.parquet
  pat2pub      not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/patent2pu

## 1. Load cached CSR (references) + journal per code + focal journal papers in range

In [2]:
%%time
# CSR references + journal per code, from dim.CSR_NPZ (cache/paper_csr.npz) and
# dim.JOURNAL_PQ (cache/paper_journal.parquet).
out_ptr, out_idx, in_ptr, in_idx, year, uni_mag = dim.load_csr()
n = len(uni_mag); print(f'CSR: {len(out_idx):,} edges, {n:,} papers')

dim.build_journal()
jr = pd.read_parquet(dim.JOURNAL_PQ, columns=['work_id', 'source_id', 'is_journal'])
pid = dim.id_to_code(jr['work_id'])
sid = dim.src_to_code(jr['source_id'])          # jour.… -> int, the source codec
isj = jr['is_journal'].fillna(False).to_numpy().astype(bool)
ok0 = (pid >= 0) & (sid >= 0)
pid, sid, isj = pid[ok0], sid[ok0], isj[ok0]
del jr; gc.collect()

# JOURNAL_PQ holds one row per publication that has ANY source -- journals, proceedings, book
# series, seminar series, preprint platforms. `is_journal` is `source_titles.type == 'journal'`,
# the analogue of the reference's `DocType == 'Journal'`.
#
# The reference builds its MAGID -> JID map from `df_MAG_J = df[df.DocType == 'Journal']`, so a
# cited work that is not journal-type has NO journal and forms no pair. Mapping every source
# here instead would put proceedings and book series into the pair counters as if they were
# journals. So `journal_by_code` is set only where
# `is_journal`; everything else stays -1.
i = np.searchsorted(uni_mag, pid); i = np.clip(i, 0, n - 1); matched = uni_mag[i] == pid

is_journal_code = np.zeros(n, np.bool_)
is_journal_code[i[matched]] = isj[matched]

journal_by_code = np.full(n, -1, np.int64)
journal_matched = matched & isj
journal_by_code[i[journal_matched]] = sid[journal_matched]

# Defensive, and cheap: the two must describe the same set of works. They can only diverge if
# JOURNAL_PQ carries a work twice with conflicting is_journal, which would make the scatter
# order-dependent -- loud here rather than a silent bias in every pair count downstream.
_bad = int(((journal_by_code >= 0) & ~is_journal_code).sum())
assert _bad == 0, (f'{_bad:,} works have a journal code but is_journal=False; '
                   f'{dim.JOURNAL_PQ} has conflicting duplicate work_id rows')
del pid, sid, isj, i, matched, journal_matched; gc.collect()
print(f'journal-mapped (cited side): {(journal_by_code >= 0).sum():,} | '
      f'journal-type (focal side): {is_journal_code.sum():,}   [must be equal]')

focal_codes = np.flatnonzero(is_journal_code).astype(np.int64)
yy = year[focal_codes]
focal_codes = focal_codes[(yy >= YEAR_RANGE[0]) & (yy <= YEAR_RANGE[1])]
print(f'focal journal papers in {YEAR_RANGE}: {len(focal_codes):,}')
# Per-year volume drives the per-year cost, so print it: it is what makes a short run
# extrapolable to the full range.
_yv, _yc = np.unique(year[focal_codes], return_counts=True)
print('focal papers per year:')
for _y, _c in zip(_yv.tolist(), _yc.tolist()):
    print(f'    {_y}  {_c:>10,}')

CSR cache present: /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
CSR: 2,141,693,663 edges, 155,441,856 papers
journal cache present: /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_journal.parquet
journal-mapped (cited side): 119,605,748 | journal-type (focal side): 119,605,748   [must be equal]
focal journal papers in (1990, 2000): 13,838,795
focal papers per year:
    1990   1,092,849
    1991   1,109,624
    1992   1,139,524
    1993   1,172,744
    1994   1,206,789
    1995   1,265,556
    1996   1,309,315
    1997   1,344,701
    1998   1,367,554
    1999   1,379,446
    2000   1,450,693


## 2. Uzzi atypicality engine — code space, one focal-year cohort at a time (verbatim from OpenAlex)

In [3]:
from itertools import combinations
from collections import defaultdict
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple
from tqdm.auto import tqdm

Pair = Tuple[int, int]
CohortEntry = Tuple[int, np.ndarray, np.ndarray]      # (focal code, reference codes, cited years)


def _journal_pair_counts(journal_codes: Iterable[int],
                         counter: Optional[Dict[Pair, int]] = None,
                         return_distinct: bool = False,
                         count: bool = True):
    """Count every unordered i<j journal pair of one reference list, WITH multiplicity.

    `[A, A, A, B, B]` -> `(A,A): 3`, `(A,B): 6`, `(B,B): 1`. The reference's `cal_J`
    increments its counter once per reference-position pair, so a journal repeated among a
    paper's references contributes quadratically; self-pairs `(A, A)` are real and kept.

    Codes < 0 (the cited work has no journal) are dropped BEFORE pairing, so such a reference
    neither pairs with another unmapped reference nor with a journal-bearing one -- `cal_J`
    requires both endpoints to be in the reference's `MAGID -> JID` map.

    Observed and null counts both go through here so they cannot drift apart.

    `count=False` enumerates the pairs and returns only the distinct set, skipping the
    counter increments -- what the per-paper aggregation pass needs, where the counts are
    already known and only the paper's own set of pairs is wanted.
    """
    codes = np.asarray(journal_codes)
    codes = codes[codes >= 0]
    if counter is None:
        counter = defaultdict(int)
    distinct: Optional[Set[Pair]] = set() if return_distinct else None
    if len(codes) >= 2:
        for a, b in combinations(codes.tolist(), 2):
            pair = (a, b) if a <= b else (b, a)
            if count:
                counter[pair] += 1
            if distinct is not None:
                distinct.add(pair)
    return (counter, distinct) if return_distinct else counter


def _sample_without_replacement(rng: np.random.Generator, n: int, k: int) -> np.ndarray:
    """`k` distinct indices out of `n`, in draw order, in O(k) rather than O(n).

    The order matters: the drawn donors are handed out to focal-paper slots sequentially.

    `np.random.choice(replace=False)` permutes the whole population, which is unaffordable
    when a donor pool holds tens of millions of edges and only a few thousand slots are drawn.
    Partial Fisher-Yates on a dict gives the same distribution at the size actually needed.
    """
    if k < 0:
        raise ValueError(f'k must be non-negative, got {k}')
    if k > n:
        raise ValueError(f'Cannot draw {k} slots without replacement from a pool of {n}')
    if k == 0:
        return np.empty(0, dtype=np.int64)
    if k == n:
        # The whole cell is focal. The reference permutes the entire cited column here, so
        # returning np.arange(n) -- the identity -- would mean no rewiring at all.
        return rng.permutation(n).astype(np.int64, copy=False)
    swap: Dict[int, int] = {}
    out = np.empty(k, np.int64)
    for i in range(k):
        j = int(rng.integers(i, n))
        out[i] = swap.get(j, j)
        swap[j] = swap.get(i, i)
    return out


def _rng_for_year(seed: int, focal_year: int) -> np.random.Generator:
    """An independent stream per focal year, so a year's result does not depend on which
    other years were in the run or on the order they were processed in. Running 2000 alone
    reproduces 2000 from a full-range run, given the same caches and parameters."""
    return np.random.default_rng(np.random.SeedSequence([int(seed), int(focal_year)]))


def _draw_null_pair_counts(cohort: Sequence[CohortEntry],
                           pool: Dict[int, np.ndarray],
                           journal_by_code: np.ndarray,
                           rng: np.random.Generator) -> Dict[Pair, int]:
    """One randomized network for one focal-year cohort -> its journal-pair counter.

    EVERY paper in `cohort` participates, whatever its observed references looked like. A
    paper whose references carry no journal contributes nothing observed, but rewiring can
    hand it journal-bearing references and the reference counts those; excluding it would
    bias every null mean downward. Selection on the observed side belongs to `work_pairs`,
    which is used only for the per-paper aggregate.

    Each cited year is drawn independently and without replacement from that
    `(citing year, cited year)` cell's donor pool, which preserves both endpoints' years.
    """
    need: Dict[int, List[Tuple[int, int]]] = defaultdict(list)   # cited year -> [(cohort i, slots)]
    for i, (_code, _refs, cited_years) in enumerate(cohort):
        uniq_years, counts = np.unique(cited_years, return_counts=True)
        for cited_year, count in zip(uniq_years.tolist(), counts.tolist()):
            need[int(cited_year)].append((i, int(count)))

    drawn: Dict[int, List[np.ndarray]] = {i: [] for i in range(len(cohort))}
    for cited_year, requests in need.items():
        donors = pool.get(cited_year)
        if donors is None or len(donors) == 0:
            # Unreachable in production: the focal papers of this cohort are themselves
            # citing papers of this year, so every cited year they use is in the cell.
            continue
        total_requested = sum(c for _, c in requests)
        if total_requested > len(donors):
            raise RuntimeError(
                f'cited year {cited_year}: {total_requested:,} focal slots requested from a '
                f'donor pool of {len(donors):,}. Focal slots are a subset of the cell, so '
                f'this means `pool` was not built from the graph the cohort came from.')
        take = donors[_sample_without_replacement(rng, len(donors), total_requested)]
        off = 0
        for i, count in requests:
            drawn[i].append(take[off:off + count])
            off += count

    counts: Dict[Pair, int] = defaultdict(int)
    for i in range(len(cohort)):
        if not drawn[i]:
            continue
        _journal_pair_counts(journal_by_code[np.concatenate(drawn[i])], counter=counts)
    return counts


def atypicality_mag(focal_codes, year, out_ptr, out_idx, journal_by_code,
                    num_shuffles=10, min_refs=2, max_refs=1000, seed=42, verbose=True):
    """Uzzi atypicality, following MAG-Atyp-Comb-1990-2000.ipynb.

    Statistically the reference's model; structurally not, because the reference materialises
    `D[(citing_year, cited_year)]` and ten rewired copies of it as Python dicts, against a
    decade of MAG. This snapshot has ~2.2e9 edges.

    1. THE NULL'S DONOR POOL IS THE WHOLE (citing year, cited year) CELL, not the focal
       cohort. The reference builds `D[(citing_year, cited_year)]` over EVERY citing paper of
       that year and shuffles the cited column inside it, so a focal paper's counterfactual
       references are drawn from everything published in the cited year that anyone citing in
       that year referenced -- including the references of non-journal papers, and including
       cited papers that carry no journal. Shuffling only within the cohort would make the
       null a statement about the cohort instead of about the literature.

       Materialising that permutation is not possible here and is not needed: a uniform
       permutation of a cell's cited column is exchangeable, so the slots belonging to focal
       papers are a simple random sample WITHOUT replacement from that cell's cited pool.
       Drawing only those slots is exact, not an approximation (validated below).

    2. EVERY ELIGIBLE FOCAL PAPER IS IN THE NULL. `cohort` is every focal paper with
       `min_refs <= len(refs) <= max_refs`, and the null draws over all of it. Having an
       observed journal pair decides only whether a paper gets an OUTPUT ROW, and that is
       settled in the second pass -- exactly as the reference, where `Ref_List[p][k]` exists
       for every rewired paper while `Paper_pair[k]` exists only for papers `cal_J` fired on.

    3. THE COHORT IS WALKED TWICE, on purpose. Holding each paper's distinct-pair set
       until the z of every pair is known costs hundreds of GB at 2010s cohort sizes and was
       measured OOM-killing the 2019+ chunks. z depends only on the pair, so pass one counts
       pairs, and pass two -- after z exists -- rebuilds one paper's set at a time to
       aggregate it. The second pass re-enumerates the observed pairs, roughly +8% runtime
       against the ten null draws that dominate the cohort, for bounded memory.

    4. PAIRS ARE COUNTED WITH MULTIPLICITY, observed and null alike, through
       `_journal_pair_counts`. The per-paper aggregate still uses the DISTINCT pairs the
       paper touches (`Paper_pair` is a set there), which is why a paper's `n_pairs` is far
       smaller than its contribution to the counters.

    Kept from the reference: `num_shuffles=10`, papers with more than `max_refs` references
    skipped, journal pairs whose null standard deviation is zero dropped (no smoothing), and
    per-paper median and 10th percentile of the z of its distinct pairs. `Z_min` is an
    addition. Randomness is seeded per focal year, so the result of a year does not depend on
    the range it was run in.

    Returns (work_df[code, Z_median, Z_10pct, Z_min, n_pairs],
             pair_df[code_1, code_2, year, Z_score]).
    """
    fy = year[focal_codes]
    order = np.argsort(fy, kind='stable')
    fc, fys = focal_codes[order], fy[order]
    uyears, starts = np.unique(fys, return_index=True)
    bounds = list(starts) + [len(fc)]

    # Papers grouped by publication year, so a citing-year cell can be assembled without
    # scanning the whole snapshot once per year.
    yorder = np.argsort(year, kind='stable')
    ysorted = year[yorder]
    uy, ustart = np.unique(ysorted, return_index=True)
    uend = list(ustart[1:]) + [len(ysorted)]
    ystart = {int(y): (int(a), int(b)) for y, a, b in zip(uy, ustart, uend)}

    # NOTE: pair_rows accumulates every (journal pair, year, z) for the whole run. At the
    # reference decade this is fine; a full 1900-2020 production run may eventually need
    # year-partitioned or streaming parquet output instead of one in-memory list.
    work_rows, pair_rows = [], []
    for gi in tqdm(range(len(uyears)), desc='atypicality by year', disable=not verbose):
        cy = int(uyears[gi])
        grp = fc[bounds[gi]:bounds[gi + 1]]
        if cy not in ystart:
            continue

        # ---- the donor pools: every edge whose citing paper was published in cy ----------
        s, e = ystart[cy]
        citing_all = yorder[s:e]
        seg = [out_idx[out_ptr[p]:out_ptr[p + 1]] for p in citing_all.tolist()]
        seg = [x for x in seg if len(x)]
        if not seg:
            continue
        dst = np.concatenate(seg)
        del seg
        dy_all = year[dst]
        o = np.argsort(dy_all, kind='stable')
        dst, dy_all = dst[o], dy_all[o]
        du, dstart = np.unique(dy_all, return_index=True)
        dend = list(dstart[1:]) + [len(dst)]
        pool = {int(y): dst[a:b] for y, a, b in zip(du, dstart, dend)}
        del dst, dy_all, o, du, dstart, dend
        gc.collect()

        # ---- COHORT: every eligible focal paper, with its references and their years ------
        # Eligibility is reference-count only. Whether the observed references carry journals
        # is decided later and does NOT remove a paper from the null.
        cohort: List[CohortEntry] = []
        for F in grp.tolist():
            a0, a1 = out_ptr[F], out_ptr[F + 1]
            k = a1 - a0
            if k < min_refs or k > max_refs:      # the reference skips >1000 outright
                continue
            refs = out_idx[a0:a1]
            cohort.append((F, refs, year[refs]))
        if not cohort:
            continue

        # ---- observed: pair counts only -- NOT one distinct-pair set per paper -----------
        # Keeping `work_pairs[F]` for every focal paper is what made the 2010s cohorts
        # unrunnable: a paper with m journal-bearing references holds up to m(m+1)/2 tuples,
        # and at 3.7M focal papers x ~50 references that is hundreds of GB. Measured: the
        # 2019-2021, 2022-2023 and 2024-2025 chunks were OOM-killed at 250 GB, and 2016-2018
        # sat at 243 GB after 13.5 h without finishing a single year.
        #
        # The sets are not needed yet. z is a property of the PAIR, so it can be computed
        # from `obs` alone; a paper's own pairs are only wanted afterwards, to aggregate the
        # z it touches. So the cohort is walked twice and the sets live one paper at a time.
        obs: Dict[Pair, int] = defaultdict(int)
        for F, refs, _cited_years in cohort:
            _journal_pair_counts(journal_by_code[refs], counter=obs)
        if not obs:
            continue                              # no observed pair -> no z to compute

        # ---- null: num_shuffles randomized networks over the WHOLE cohort -----------------
        year_rng = _rng_for_year(seed, cy)
        null_counts = [_draw_null_pair_counts(cohort, pool, journal_by_code, year_rng)
                       for _ in range(num_shuffles)]

        # ---- z per journal pair; the reference drops std == 0 rather than smoothing -------
        zmap: Dict[Pair, float] = {}
        for p, observed in obs.items():
            vals = np.array([nc.get(p, 0) for nc in null_counts], float)
            mu, sd = vals.mean(), vals.std()      # population std, ddof=0, as in the reference
            if sd == 0.0:
                continue
            z = (observed - mu) / sd
            zmap[p] = z
            pair_rows.append((p[0], p[1], cy, z))

        # ---- per-paper aggregate: the SECOND pass over the cohort -------------------------
        # Each paper's distinct pairs are rebuilt, used, and dropped before the next paper,
        # so peak memory here is one paper's set rather than the whole cohort's. The pairs
        # are enumerated by the same helper as the observed pass, so "the distinct pairs this
        # paper touches" cannot come to mean something different between the two.
        for F, refs, _cited_years in cohort:
            _, distinct = _journal_pair_counts(journal_by_code[refs], return_distinct=True,
                                               count=False)
            if not distinct:
                continue                          # no observed pair -> not in the output
            zs = [zmap[p] for p in distinct if p in zmap]
            if zs:
                arr = np.asarray(zs)
                work_rows.append((F, float(np.quantile(arr, .5)), float(np.quantile(arr, .1)),
                                  float(arr.min()), len(arr)))
        del cohort, obs, null_counts, zmap, pool
        gc.collect()

    work_df = pd.DataFrame(work_rows,
                           columns=['code', 'Z_median', 'Z_10pct', 'Z_min', 'n_pairs'])
    pair_df = pd.DataFrame(pair_rows, columns=['code_1', 'code_2', 'year', 'Z_score'])
    return work_df, pair_df


print('atypicality engine ready — MAG-Atyp-Comb null (cell-wide donor pool, whole cohort in '
      'the null, pairs counted with multiplicity, per-year seeding)')

atypicality engine ready — MAG-Atyp-Comb null (cell-wide donor pool, whole cohort in the null, pairs counted with multiplicity, per-year seeding)


## 3. Compute (per-year Uzzi) and save both outputs

In [4]:
%%time
work_df, pair_df = atypicality_mag(
    focal_codes, year, out_ptr, out_idx, journal_by_code,
    num_shuffles=NUM_SHUFFLES, min_refs=MIN_REFS, max_refs=MAX_REFS, seed=SEED)
work_df['paper_id'] = dim.code_to_id(uni_mag[work_df['code'].to_numpy()])
work_df = work_df[['paper_id', 'Z_median', 'Z_10pct', 'Z_min', 'n_pairs']]
work_df.to_parquet(f'{OUT}/paper_z_score{SUFFIX}.parquet', index=False)
pair_df.to_parquet(f'{OUT}/z_score_pair{SUFFIX}.parquet', index=False)
print(f'WROTE {OUT}/paper_z_score{SUFFIX}.parquet  ({len(work_df):,} papers)')
print(f'WROTE {OUT}/z_score_pair{SUFFIX}.parquet   ({len(pair_df):,} (pair,year) rows)')
display(work_df.head(10)); display(pair_df.head(10))
print(work_df[['Z_median', 'Z_10pct', 'Z_min', 'n_pairs']].describe().round(4).to_string())

WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_z_score_1990_2000.parquet  (6,193,175 papers)
WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/z_score_pair_1990_2000.parquet   (34,061,836 (pair,year) rows)


,paper_id,Z_median,Z_10pct,Z_min,n_pairs
0,pub.1000000022,5.338325,-5.058893,-26.065594,439
1,pub.1000000061,445.961006,445.961006,445.961006,1
2,pub.1000000486,32.482872,32.482872,32.482872,1
3,pub.1000000507,907.308985,617.370352,544.885694,3
4,pub.1000000515,116.389906,1.316725,-0.968425,30
5,pub.1000000641,104.654917,19.133763,9.645981,18
6,pub.1000000806,16.991696,-7.789047,-35.339523,71
7,pub.1000000858,8.710510,-1.575943,-7.497473,30
8,pub.1000001191,362.306953,191.958097,149.370883,2
9,pub.1000001217,39.586407,7.105766,7.087588,10


,code_1,code_2,year,Z_score
0,1013101,1062273,1990,174.576354
1,1013101,1018654,1990,8.233288
2,1013101,1017390,1990,10.237438
3,1013101,1013101,1990,985.780582
4,1013101,1036281,1990,35.443819
5,1013101,1017255,1990,363.644694
6,1013101,1014535,1990,25.736564
7,1012103,1013101,1990,-3.887710
8,1013101,1019073,1990,180.630281
9,1013101,1077219,1990,42.146179


           Z_median       Z_10pct         Z_min       n_pairs
count  6.193175e+06  6.193175e+06  6.193175e+06  6.193175e+06
mean   1.931049e+02  7.390270e+01  4.501540e+01  7.329760e+01
std    5.293349e+02  4.449398e+02  4.409599e+02  2.060349e+02
min   -2.043279e+02 -2.310163e+02 -3.025418e+02  1.000000e+00
25%    2.456010e+01 -4.262000e+00 -2.647910e+01  9.000000e+00
50%    6.597460e+01  2.673200e+00 -3.979000e+00  3.000000e+01
75%    1.731253e+02  2.592900e+01  8.224900e+00  8.200000e+01
max    2.275168e+05  2.275168e+05  2.275168e+05  5.268900e+04


## Validation against MAG null-model semantics

Five assertions on a **toy graph** — no CSR, no cache, a few hundred microseconds each. They pin the
four places where this implementation could silently drift from `MAG-Atyp-Comb-1990-2000.ipynb`:
pair multiplicity, the complete-pool permutation, the participation of observed-invalid papers in the
null, and the equivalence of partial sampling to an explicit full-column permutation. Test 3 is the
regression test for the null-participation bug: it fails against any version that filters the null
assignments by `F in work_pairs`.

The toy `(citing year 2000, cited year 1990)` cell:

| citing | focal? | cell slots | cited | journal of cited |
|---|---|---|---|---|
| `P0` | yes | 0, 1 | `c0`, `c1` | none, none |
| `P1` | yes | 2, 3 | `c2`, `c3` | `A`, `B` |
| `P2` | no  | 4, 5 | `c4`, `c5` | `A`, `B` |

`P0` has two references and neither carries a journal, so it has no observed pair and is absent from
`work_pairs` — but it is in `cohort`, it draws two donors out of the six, and it can therefore
contribute `(A, B)` to a null network. `P2` is not focal: it donates its slots to the pool but its own
rewired references are never counted.

In [5]:
# Validation of the null-model semantics on a toy graph. Plain asserts, no pytest, no cache.
_A, _B = 100, 200                       # two journal codes
_N_CODES = 20
_jbc = np.full(_N_CODES, -1, np.int64)  # journal_by_code for the toy graph
_jbc[12] = _A; _jbc[13] = _B            # c2, c3
_jbc[14] = _A; _jbc[15] = _B            # c4, c5   (c0=10, c1=11 stay unmapped)

# One focal-year cohort: (focal code, reference codes, cited years).
_P0 = (0, np.array([10, 11]), np.array([1990, 1990]))
_P1 = (1, np.array([12, 13]), np.array([1990, 1990]))
_cohort = [_P0, _P1]
# The complete (citing 2000, cited 1990) donor cell, including non-focal P2's two edges.
_cell_cited = np.array([10, 11, 12, 13, 14, 15])
_pool = {1990: _cell_cited}
_focal_slots = [np.array([0, 1]), np.array([2, 3])]     # P0 owns cell slots 0-1, P1 owns 2-3

# ---- Test 1: pair multiplicity ---------------------------------------------------------------
_counts, _distinct = _journal_pair_counts([_A, _A, _A, _B, _B], return_distinct=True)
assert _counts[(_A, _A)] == 3, _counts
assert _counts[(_A, _B)] == 6, _counts
assert _counts[(_B, _B)] == 1, _counts
assert len(_distinct) == 3, _distinct
assert dict(_journal_pair_counts([_A, -1, _B])) == {(_A, _B): 1}, 'unmapped code must not pair'
assert dict(_journal_pair_counts([_A])) == {}, 'a single reference forms no pair'
print('1 PASS  pair multiplicity: [A,A,A,B,B] -> (A,A)=3 (A,B)=6 (B,B)=1, 3 distinct')

# ---- Test 2: k == n must be a real permutation, not the identity ------------------------------
_n = 7
_perms = [_sample_without_replacement(np.random.default_rng(s), _n, _n) for s in range(8)]
for _p in _perms:
    assert sorted(_p.tolist()) == list(range(_n)), 'k == n must return every index exactly once'
    assert _p.dtype == np.int64
assert any((_p != np.arange(_n)).any() for _p in _perms), 'k == n returned only the identity'
try:
    _sample_without_replacement(np.random.default_rng(0), 3, 4)
    raise AssertionError('k > n must raise ValueError')
except ValueError:
    pass
_empty = _sample_without_replacement(np.random.default_rng(0), 5, 0)
assert _empty.shape == (0,) and _empty.dtype == np.int64
print('2 PASS  k==n permutes the whole pool; k>n raises; k==0 is an empty int64 array')

# ---- Test 3: a paper with no observed journal pair still enters the null -----------------------
_obs, _work_pairs = defaultdict(int), {}
for _code, _refs, _ in _cohort:
    _, _d = _journal_pair_counts(_jbc[_refs], counter=_obs, return_distinct=True)
    if _d:
        _work_pairs[_code] = _d
assert 0 not in _work_pairs, 'P0 must have no observed journal pair'
assert _work_pairs[1] == {(_A, _B)}, 'P1 must observe exactly (A, B)'

_rng = _rng_for_year(SEED, 2000)
_full_max = max(_draw_null_pair_counts(_cohort, _pool, _jbc, _rng)[(_A, _B)] for _ in range(300))
# The legacy behaviour: restrict the null to papers that have an observed pair.
_legacy_cohort = [c for c in _cohort if c[0] in _work_pairs]
_rng = _rng_for_year(SEED, 2000)
_legacy_max = max(_draw_null_pair_counts(_legacy_cohort, _pool, _jbc, _rng)[(_A, _B)]
                  for _ in range(300))
assert _legacy_max <= 1, f'only P1 can contribute, so (A,B) cannot exceed 1; got {_legacy_max}'
assert _full_max >= 2, ('P0 never contributed to the null -- the cohort is being filtered by '
                        'work_pairs somewhere')
print(f'3 PASS  null participation: max (A,B) per draw = {_full_max} with the whole cohort, '
      f'{_legacy_max} when filtered by work_pairs')

# ---- Test 4: partial sampling == explicit full-column permutation (distributionally) ----------
def _explicit_permutation_null(cell_cited, focal_slots, journal_by_code, rng):
    """Validation-only baseline: permute the cell's ENTIRE cited column, as the reference's
    `random.shuffle(temp)` does, then read the focal papers' own slots back off it. Correct,
    and O(cell size) per draw -- which is exactly why the engine does not do this."""
    rewired = np.asarray(cell_cited)[rng.permutation(len(cell_cited))]
    counts = defaultdict(int)
    for slots in focal_slots:
        _journal_pair_counts(journal_by_code[rewired[slots]], counter=counts)
    return counts

_N_MC = 8000
_pairs_watched = [(_A, _A), (_A, _B), (_B, _B)]
_rng_a = np.random.default_rng(20240001)
_rng_b = np.random.default_rng(20240002)
_base = [_explicit_permutation_null(_cell_cited, _focal_slots, _jbc, _rng_a) for _ in range(_N_MC)]
_fast = [_draw_null_pair_counts(_cohort, _pool, _jbc, _rng_b) for _ in range(_N_MC)]
print(f'4 ...   {_N_MC:,} toy draws each; null means per journal pair')
for _p in _pairs_watched:
    _mb = float(np.mean([c[_p] for c in _base]))
    _mf = float(np.mean([c[_p] for c in _fast]))
    print(f'        {_p}  explicit permutation {_mb:.4f}   partial sampling {_mf:.4f}   '
          f'|d| {abs(_mb - _mf):.4f}')
    assert abs(_mb - _mf) < 0.05, (f'{_p}: null means differ by {abs(_mb - _mf):.4f}; the '
                                   f'partial sampler is not equivalent to a full permutation')
print('4 PASS  partial sampling reproduces the full-permutation null within Monte Carlo error')

# ---- Test 5: a focal year's stream does not depend on the range it was run in -------------------
_alone = _rng_for_year(SEED, 2000).integers(0, 10**9, size=6)
_ = [_rng_for_year(SEED, _y).integers(0, 10**9, size=6) for _y in range(1990, 2000)]
_after = _rng_for_year(SEED, 2000).integers(0, 10**9, size=6)
assert (_alone == _after).all(), 'per-year streams must not depend on processing order'
# The comparisons are bound to names first so that no line begins with '!=':
# run_notebook.py strips lines starting with '!' as shell escapes, which would silently
# delete a continuation line and leave a passing-but-meaningless assert behind.
_y2001  = _rng_for_year(SEED, 2001).integers(0, 10**9, size=6)
_seed43 = _rng_for_year(SEED + 1, 2000).integers(0, 10**9, size=6)
assert (_alone != _y2001).any(), 'different focal years must give different streams'
assert (_alone != _seed43).any(), 'different global seeds must give different streams'
print('5 PASS  _rng_for_year(seed, year) is independent of processing order and year range')
print('        (full-vs-partial equality of results also needs identical caches and parameters)')

del _A, _B, _N_CODES, _jbc, _P0, _P1, _cohort, _cell_cited, _pool, _focal_slots
del _counts, _distinct, _perms, _empty, _obs, _work_pairs, _rng, _full_max, _legacy_cohort
del _legacy_max, _N_MC, _pairs_watched, _rng_a, _rng_b, _base, _fast, _alone, _after
del _y2001, _seed43
print('\nALL VALIDATION TESTS PASSED')

1 PASS  pair multiplicity: [A,A,A,B,B] -> (A,A)=3 (A,B)=6 (B,B)=1, 3 distinct
2 PASS  k==n permutes the whole pool; k>n raises; k==0 is an empty int64 array
3 PASS  null participation: max (A,B) per draw = 2 with the whole cohort, 1 when filtered by work_pairs
4 ...   8,000 toy draws each; null means per journal pair
        (100, 100)  explicit permutation 0.1344   partial sampling 0.1323   |d| 0.0021
        (100, 200)  explicit permutation 0.5355   partial sampling 0.5371   |d| 0.0016
        (200, 200)  explicit permutation 0.1310   partial sampling 0.1369   |d| 0.0059
4 PASS  partial sampling reproduces the full-permutation null within Monte Carlo error
5 PASS  _rng_for_year(seed, year) is independent of processing order and year range
        (full-vs-partial equality of results also needs identical caches and parameters)

ALL VALIDATION TESTS PASSED
